In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

repo_root = Path().resolve().parents[0]  # notebooks/ -> repo root
sys.path.insert(0, str(repo_root))

In [3]:
import pickle as pkl
from src.dataset.dataset import TracesDataset

In [8]:
with open(r"../data/raw/gtea_target.pkl", "rb") as f:
    gtea_target = pkl.load(f)
with open(r"../data/raw/gtea_softmax.pkl", "rb") as f:
    gtea_source = pkl.load(f)
with open(r"../data/processed/gtea_unified.pkl", "rb") as f:
    gtea_unified = pkl.load(f)

In [9]:
gtea_target[0].shape, gtea_source[0].shape

((1643,), (1, 11, 1643))

In [10]:
tds = TracesDataset(gtea_target, gtea_source)

In [11]:
batch = tds[:5]

In [12]:
[x.shape for x in batch[0]]

[torch.Size([1643]),
 torch.Size([1643]),
 torch.Size([1643]),
 torch.Size([1643]),
 torch.Size([943])]

In [13]:
[x.shape for x in batch[1]]

[torch.Size([1643, 11]),
 torch.Size([1643, 11]),
 torch.Size([1643, 11]),
 torch.Size([1643, 11]),
 torch.Size([943, 11])]

In [14]:
from torch.nn.utils.rnn import pad_sequence
from torch.nn.functional import one_hot

pad_value = tds.n_classes

label_batch = one_hot(pad_sequence(batch[0], batch_first=True, padding_value=pad_value), num_classes=max(tds.n_classes, pad_value + 1))

In [15]:
import torch.nn.functional as F

output_constant = pad_sequence(batch[1], batch_first=True, padding_value=0)
output_constant = F.pad(output_constant, (0, label_batch.shape[-1] - output_constant.shape[-1]), value=0.0)
output_constant.shape

torch.Size([5, 1643, 12])

In [16]:
from torch.utils.data import DataLoader
from src.dataset.dataset import collate_traces_batch
from functools import partial

num_channels = 11
collate_one_hot = partial(collate_traces_batch, padding_value=7, final_channels=num_channels, one_hot_labels=True)

dataloader = DataLoader(tds, batch_size=5, shuffle=True, collate_fn=collate_one_hot)

In [17]:
next(iter(dataloader))

(tensor([[[0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          ...,
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1]],
 
         [[0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          ...,
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1]],
 
         [[0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          ...,
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1]],
 
         [[0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          ...,
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0, 0, 1]],
 
         [[0, 0, 0,  ..., 0, 0, 1],
          [0, 0, 0,  ..., 0,

In [20]:
next(iter(dataloader))[0].shape

torch.Size([5, 2009, 122])